In [16]:
# Video Verification

In [17]:
from pathlib import Path
import csv
import pickle
from datetime import datetime

import cv2
import numpy as np

# Select the original input video.
VIDEO_PATH = Path(
    r"C:\Users\user\Desktop\M2\Computer Vision and Image Processing\Major project\eye_videos\eye_videos\test\test01_asian_female_anchor_12s.mp4"
)

# Use the six-class model exported by step2.
MODEL_PATH = Path(
    r"C:\Users\user\Desktop\M2\Computer Vision and Image Processing\Major project"
    r"\mouth_viseme_training_runs\20261026_224845_2d72a7"
    r"\mouth_viseme_hog_svm.pkl"
)

In [18]:
FEATURE_PARAMS = dict(
    maxCorners=150,
    qualityLevel=0.01,
    minDistance=7,
    blockSize=7
)

LK_PARAMS = dict(
    winSize=(21, 21),
    maxLevel=3,
    criteria=(
        cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
        30,
        0.01
    )
)

MIN_POINTS = 10
FB_THRESHOLD = 1.5
RANSAC_THRESHOLD = 3.0
REPLENISH_THRESHOLD = 80
PANEL_WIDTH = 320


def mouth_select_roi(frame, title):
    """Select a region manually."""

    x, y, w, h = cv2.selectROI(
        title,
        frame,
        showCrosshair=True,
        fromCenter=False
    )
    cv2.destroyWindow(title)

    if w < 8 or h < 8:
        raise ValueError("Region is too small or selection was canceled.")

    return int(x), int(y), int(w), int(h)


def mouth_rect_points(rect):
    """Convert a rectangle to four corners."""

    x, y, w, h = rect

    return np.float32([
        [x, y],
        [x + w - 1, y],
        [x + w - 1, y + h - 1],
        [x, y + h - 1]
    ]).reshape(-1, 1, 2)


def mouth_initialize(frame):
    """Select the head and mouth, then initialize head tracking points."""

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    head_rect = mouth_select_roi(
        frame,
        "Select whole head, then Enter"
    )

    mouth_rect = mouth_select_roi(
        frame,
        "Select full mouth with some margin, then Enter"
    )

    mask = np.zeros(gray.shape, dtype=np.uint8)
    x, y, w, h = head_rect
    mask[y:y + h, x:x + w] = 255

    points = cv2.goodFeaturesToTrack(
        gray,
        mask=mask,
        **FEATURE_PARAMS
    )

    if points is None or len(points) < MIN_POINTS:
        raise ValueError("Too few head tracking points. Select again.")

    return gray, points, mouth_rect_points(head_rect), mouth_rect


def mouth_track_points(previous_gray, gray, points, reference):
    """LK optical flow with a forward-backward check."""

    current, status, _ = cv2.calcOpticalFlowPyrLK(
        previous_gray,
        gray,
        points,
        None,
        **LK_PARAMS
    )

    if current is None or status is None:
        return None, None

    valid = (
        (status.ravel() == 1)
        & np.isfinite(current.reshape(-1, 2)).all(axis=1)
    )

    if not valid.any():
        return None, None

    old = points[valid]
    current = current[valid]
    reference = reference[valid]

    backward, back_status, _ = cv2.calcOpticalFlowPyrLK(
        gray,
        previous_gray,
        current,
        None,
        **LK_PARAMS
    )

    if backward is None or back_status is None:
        return None, None

    error = np.linalg.norm(
        old.reshape(-1, 2) - backward.reshape(-1, 2),
        axis=1
    )

    height, width = gray.shape
    xy = current.reshape(-1, 2)

    reliable = (
        (back_status.ravel() == 1)
        & np.isfinite(error)
        & (error < FB_THRESHOLD)
        & (xy[:, 0] >= 0)
        & (xy[:, 0] < width)
        & (xy[:, 1] >= 0)
        & (xy[:, 1] < height)
    )

    return current[reliable], reference[reliable]


def mouth_replenish(gray, points, reference, transform, corners):
    """Add corners inside the head region and map them to reference coordinates."""

    if len(points) >= REPLENISH_THRESHOLD:
        return points, reference

    mask = np.zeros(gray.shape, dtype=np.uint8)

    cv2.fillConvexPoly(
        mask,
        np.rint(corners).astype(np.int32).reshape(-1, 2),
        255
    )

    mask = cv2.erode(
        mask,
        np.ones((9, 9), dtype=np.uint8)
    )

    for point in points.reshape(-1, 2):
        cv2.circle(
            mask,
            tuple(np.rint(point).astype(int)),
            10,
            0,
            -1
        )

    params = FEATURE_PARAMS.copy()
    params["maxCorners"] = FEATURE_PARAMS["maxCorners"] - len(points)

    if params["maxCorners"] <= 0:
        return points, reference

    new_points = cv2.goodFeaturesToTrack(
        gray,
        mask=mask,
        **params
    )

    if new_points is None:
        return points, reference

    new_reference = cv2.perspectiveTransform(
        new_points,
        np.linalg.inv(transform)
    )

    valid = np.isfinite(
        new_reference.reshape(-1, 2)
    ).all(axis=1)

    return (
        np.concatenate([points, new_points[valid]], axis=0),
        np.concatenate([reference, new_reference[valid]], axis=0)
    )


def mouth_get_crop(aligned, valid_mask, rect):
    """
    Crop the fixed mouth region after head alignment.
    valid checks complete image coverage, not correct mouth location.
    """

    x, y, w, h = rect
    height, width = aligned.shape

    if (
        x < 0 or y < 0
        or x + w > width
        or y + h > height
    ):
        return None, False

    crop = aligned[y:y + h, x:x + w]
    crop_mask = valid_mask[y:y + h, x:x + w]

    valid = (
        crop.shape == (h, w)
        and crop_mask.shape == (h, w)
        and crop.size > 0
        and np.all(crop_mask > 250)
    )

    return crop, bool(valid)


def mouth_classify(crop, model, hog, size, labels):
    """Use the same grayscale, resize and HOG steps as step2."""

    interpolation = (
        cv2.INTER_AREA
        if crop.shape[1] >= size[0]
        and crop.shape[0] >= size[1]
        else cv2.INTER_LINEAR
    )

    resized = cv2.resize(
        crop,
        size,
        interpolation=interpolation
    )

    features = hog.compute(resized).reshape(1, -1).astype(np.float32)
    predicted_id = int(model.predict(features)[0])

    return labels[predicted_id]


def mouth_draw_preview(panel, crop):
    """Display the original gray crop on the right without stretching."""

    available_height = panel.shape[0] - 115

    if crop is None or crop.size == 0 or available_height <= 0:
        return

    scale = min(
        (PANEL_WIDTH - 20) / crop.shape[1],
        available_height / crop.shape[0]
    )

    size = (
        max(1, int(crop.shape[1] * scale)),
        max(1, int(crop.shape[0] * scale))
    )

    preview = cv2.resize(crop, size)
    preview = cv2.cvtColor(preview, cv2.COLOR_GRAY2BGR)

    panel[
        105:105 + size[1],
        10:10 + size[0]
    ] = preview

In [19]:
def mouth_run_video(video_path, model_bundle=None):
    """
    model_bundle=None: sampling mode, return original mouth crops.
    With model_bundle: prediction mode, display HOG + SVM predictions.

    Space: pause or resume
    R: reselect the mouth, effective from the next frame
    Q / Esc: quit
    """

    video_path = Path(video_path)
    prediction_mode = model_bundle is not None

    model = None
    hog = None
    size = None
    labels = None

    if prediction_mode:
        model = model_bundle["model"]
        size = tuple(model_bundle["mouth_size"])
        hog = cv2.HOGDescriptor(**model_bundle["hog_params"])

        labels = {
            int(value): key
            for key, value in model_bundle["label_to_id"].items()
        }

        if hog.getDescriptorSize() != model.n_features_in_:
            raise ValueError("HOG feature size does not match the model.")

    save_results = (
        input("Save the result video and per-frame CSV? Enter y to save: ")
        .strip().lower() == "y"
    )

    cap = cv2.VideoCapture(str(video_path))
    writer = None
    csv_file = None
    output_dir = None
    samples = []
    saved_frames = 0

    window = "Mouth | Space pause | R reselect | Q quit"

    try:
        if not cap.isOpened():
            raise FileNotFoundError(f"Cannot open video: {video_path}")

        success, first_frame = cap.read()

        if not success:
            raise RuntimeError("Cannot read the first frame.")

        height, width = first_frame.shape[:2]

        if height < 240:
            raise ValueError("Video height is too small. Use a higher-resolution video.")

        fps = cap.get(cv2.CAP_PROP_FPS)

        if not np.isfinite(fps) or fps <= 0:
            fps = 30.0

        delay = max(1, int(round(1000 / fps)))

        previous_gray, points, head_corners, mouth_rect = (
            mouth_initialize(first_frame)
        )
        reference = points.copy()

        if save_results:
            run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            folder = (
                "mouth_prediction_runs"
                if prediction_mode
                else "mouth_region_runs"
            )

            output_dir = video_path.parent / folder / run_id
            output_dir.mkdir(parents=True, exist_ok=False)

            writer = cv2.VideoWriter(
                str(output_dir / "mouth_result.mp4"),
                cv2.VideoWriter_fourcc(*"mp4v"),
                fps,
                (width + PANEL_WIDTH, height)
            )

            if not writer.isOpened():
                raise RuntimeError("Cannot create output video.")

            csv_file = open(
                output_dir / "mouth_frames.csv",
                "w",
                newline="",
                encoding="utf-8-sig"
            )

            fields = [
                "frame_index", "time_s", "mouth",
                "head_tracking_ok", "localization",
                "crop_valid", "prediction",
                "roi_x", "roi_y", "roi_w", "roi_h",
                "manual_revision"
            ]

            csv_writer = csv.DictWriter(csv_file, fieldnames=fields)
            csv_writer.writeheader()

        cv2.namedWindow(window, cv2.WINDOW_NORMAL)

        frame = first_frame
        frame_index = 0
        tracking_active = True
        manual_revision = 0

        while True:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            output = frame.copy()
            panel = np.zeros(
                (height, PANEL_WIDTH, 3),
                dtype=np.uint8
            )

            transform = None
            aligned = None
            crop = None
            crop_valid = False
            prediction = "unknown"
            localization = "LOST"

            if frame_index == 0:
                transform = np.eye(3, dtype=np.float64)

            elif tracking_active:
                current, matched_reference = mouth_track_points(
                    previous_gray,
                    gray,
                    points,
                    reference
                )

                if current is None or len(current) < MIN_POINTS:
                    tracking_active = False

                else:
                    affine, status = cv2.estimateAffinePartial2D(
                        matched_reference.reshape(-1, 2),
                        current.reshape(-1, 2),
                        method=cv2.RANSAC,
                        ransacReprojThreshold=RANSAC_THRESHOLD,
                        maxIters=2000,
                        confidence=0.99,
                        refineIters=10
                    )

                    if (
                        affine is None
                        or status is None
                        or not np.isfinite(affine).all()
                    ):
                        tracking_active = False

                    else:
                        inliers = status.ravel().astype(bool)

                        if np.count_nonzero(inliers) < MIN_POINTS:
                            tracking_active = False

                        else:
                            transform = np.vstack([
                                affine,
                                [0.0, 0.0, 1.0]
                            ])

                            points = current[inliers].copy()
                            reference = matched_reference[inliers].copy()

            if tracking_active:
                scale = np.linalg.norm(transform[:2, 0])

                if not 0.05 < scale < 20:
                    tracking_active = False

            if tracking_active:
                corners = cv2.perspectiveTransform(
                    head_corners,
                    transform
                )

                xy = corners.reshape(-1, 2)

                geometry_valid = (
                    np.isfinite(xy).all()
                    and (np.abs(xy[:, 0]) < 3 * width).all()
                    and (np.abs(xy[:, 1]) < 3 * height).all()
                )

                if geometry_valid:
                    area = abs(
                        cv2.contourArea(xy.astype(np.float32))
                    )
                    geometry_valid = 100 < area < width * height

                if not geometry_valid:
                    tracking_active = False

            if tracking_active:
                points, reference = mouth_replenish(
                    gray,
                    points,
                    reference,
                    transform,
                    corners
                )

                inverse = np.linalg.inv(transform)

                aligned = cv2.warpAffine(
                    gray,
                    inverse[:2],
                    (width, height),
                    flags=cv2.INTER_LINEAR,
                    borderMode=cv2.BORDER_CONSTANT,
                    borderValue=0
                )

                valid_mask = cv2.warpAffine(
                    np.full(gray.shape, 255, dtype=np.uint8),
                    inverse[:2],
                    (width, height),
                    flags=cv2.INTER_NEAREST,
                    borderMode=cv2.BORDER_CONSTANT,
                    borderValue=0
                )

                crop, crop_valid = mouth_get_crop(
                    aligned,
                    valid_mask,
                    mouth_rect
                )

                localization = "HEAD_ROI" if crop_valid else "INVALID"

                if crop_valid:
                    if prediction_mode:
                        prediction = mouth_classify(
                            crop, model, hog, size, labels
                        )
                    else:
                        prediction = "unlabeled"

                cv2.polylines(
                    output,
                    [np.rint(corners).astype(np.int32)],
                    True,
                    (255, 255, 0),
                    2
                )

                color = (
                    (0, 255, 0) if prediction == "open"
                    else (0, 0, 255) if prediction == "closed"
                    else (255, 255, 0) if crop_valid
                    else (0, 165, 255)
                )

                polygon = cv2.perspectiveTransform(
                    mouth_rect_points(mouth_rect),
                    transform
                )

                cv2.polylines(
                    output,
                    [np.rint(polygon).astype(np.int32)],
                    True,
                    color,
                    2
                )

            else:
                cv2.putText(
                    output,
                    "Head tracking lost. Rerun this cell.",
                    (15, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.55,
                    (0, 0, 255),
                    2
                )

            lines = [
                f"Frame: {frame_index}",
                f"State: {prediction.upper()}",
                f"{localization} | R: reselect",
                "Check full lips before labeling"
            ]

            for line_index, text in enumerate(lines):
                cv2.putText(
                    panel,
                    text,
                    (10, 23 + 24 * line_index),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45,
                    (255, 255, 255),
                    1
                )

            if crop_valid:
                mouth_draw_preview(panel, crop)

            combined = np.hstack([output, panel])

            # Save the displayed frame. Manual changes apply from the next frame
            if writer is not None:
                writer.write(combined)
                saved_frames += 1

            if csv_file is not None:
                x, y, w, h = mouth_rect

                csv_writer.writerow({
                    "frame_index": frame_index,
                    "time_s": frame_index / fps,
                    "mouth": "mouth",
                    "head_tracking_ok": tracking_active,
                    "localization": localization,
                    "crop_valid": crop_valid,
                    "prediction": prediction,
                    "roi_x": x,
                    "roi_y": y,
                    "roi_w": w,
                    "roi_h": h,
                    "manual_revision": manual_revision
                })

            cv2.imshow(window, combined)
            key = cv2.waitKey(delay) & 0xFF

            if key == ord(" "):
                while True:
                    key = cv2.waitKey(30) & 0xFF

                    if key in (
                        ord(" "), ord("r"), ord("R"),
                        ord("q"), 27
                    ):
                        break

                    if cv2.getWindowProperty(
                        window, cv2.WND_PROP_VISIBLE
                    ) < 1:
                        key = ord("q")
                        break

            reselected = False

            if key in (ord("r"), ord("R")) and tracking_active:
                selection_frame = cv2.cvtColor(
                    aligned,
                    cv2.COLOR_GRAY2BGR
                )

                mouth_rect = mouth_select_roi(
                    selection_frame,
                    "Reselect full mouth, then Enter"
                )

                manual_revision += 1
                reselected = True

                print(
                    f"Mouth region corrected at frame {frame_index}. "
                    "Applies from the next frame. Skip sampling this frame."
                )

            # Skip the old crop when the user corrects the region
            if (
                not prediction_mode
                and crop_valid
                and not reselected
            ):
                samples.append({
                    "frame_index": frame_index,
                    "image": crop.copy(),
                    # No template matching runs here. Do not mark it as matched
                    "matched": False,
                    "score": float("nan"),
                    "source_video": str(video_path.resolve())
                })

            if key in (ord("q"), 27):
                break

            if cv2.getWindowProperty(
                window, cv2.WND_PROP_VISIBLE
            ) < 1:
                break

            previous_gray = gray.copy()
            success, frame = cap.read()

            if not success:
                print("Video processing complete.")
                break

            # Use actual video frame indices to avoid duplicate sample filenames
            frame_index += 1

    finally:
        cap.release()

        if writer is not None:
            writer.release()

        if csv_file is not None:
            csv_file.close()

        cv2.destroyAllWindows()

        if output_dir is not None:
            print("Output folder: ", output_dir)
            print("Saved frames: ", saved_frames)

    if not prediction_mode:
        print("Mouth crops collected: ", len(samples))

    return samples

In [20]:
# Use the VIDEO_PATH and MODEL_PATH set above
# Load the six-class mouth model and its label map

if not VIDEO_PATH.is_file():
    raise FileNotFoundError(
        f"Input video not found: {VIDEO_PATH}"
    )

if not MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"Set the path to the six-class model: {MODEL_PATH}"
    )

# Load → validate the task and labels → run the existing video loop.
with open(MODEL_PATH, "rb") as file:
    mouth_bundle = pickle.load(file)

expected_mapping = {
    "closed": 0,
    "a": 1,
    "i": 2,
    "u": 3,
    "e": 4,
    "o": 5
}

if mouth_bundle.get("task") != "mouth_viseme_6_v1":
    raise ValueError(
        "The selected model is not the six-class model. "
        "Do not use the old open/closed model."
    )

if mouth_bundle.get("label_to_id") != expected_mapping:
    raise ValueError(
        "The model label map does not match: "
        f"{mouth_bundle.get('label_to_id')}"
    )

# Check the classes learned by SVM, not just the label dictionary.
learned_classes = set(
    np.asarray(
        mouth_bundle["model"].classes_
    ).tolist()
)

if learned_classes != set(expected_mapping.values()):
    raise ValueError(
        "The model did not learn all six classes. Check the training data."
    )

print(
    "Six-class mouth model loaded. Classes:",
    list(expected_mapping)
)

print(
    "A/I/U/E/O are predicted mouth shapes. "
    "UNKNOWN means the crop or tracking is invalid."
)

print(
    "Red boxes show a closed mouth. Cyan boxes show vowel shapes. "
    "See State for the exact class."
)

mouth_run_video(
    VIDEO_PATH,
    model_bundle=mouth_bundle
);

Six-class mouth model loaded. Classes: ['closed', 'a', 'i', 'u', 'e', 'o']
A/I/U/E/O are predicted mouth shapes. UNKNOWN means the crop or tracking is invalid.
Red boxes show a closed mouth. Cyan boxes show vowel shapes. See State for the exact class.


Save the result video and per-frame CSV? Enter y to save:  y


Video processing complete.
Output folder:  C:\Users\user\Desktop\M2\Computer Vision and Image Processing\Major project\eye_videos\eye_videos\test\mouth_prediction_runs\20261026_225104_994812
Saved frames:  360
